# Multi-Agency Ridership Data Standardization and Stop-Level Aggregation Workflow


This notebook processes ridership data from multiple transit agencies to create representative stop-level datasets for weekdays, Saturdays, and Sundays. For each agency, the workflow generally includes:

- Adding day type based on the service date (Weekday, Saturday, Sunday), with holidays normalized or removed as needed.
- Cleaning and renaming columns to ensure consistency across datasets (stop IDs, stop names, and ridership fields).
- Summing boardings and alightings across directions at each stop, and aggregating multiple time periods where applicable.
- Calculating weighted averages when ridership is reported over multiple periods, using the number of service days as weights.
- Setting a ridership basis flag to indicate that values represent reported daily boardings and/or alightings.
- Computing final average ridership per stop, and day type, producing clean, representative stop-level datasets ready for modeling, analysis, or comparison across agencies.

This approach standardizes ridership data across agencies with differing data formats, missing values, and service coverage, enabling consistent cross-agency analysis.

In [1]:
pip install shared_utils

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importing necessary package 
import pandas as pd 
import geopandas as gpd
import numpy as np
import google.auth
import os
import gcsfs
from calitp_data_analysis.sql import get_engine
from calitp_data_analysis import utils
db_engine = get_engine()
credentials, project = google.auth.default()
from pandas.tseries.holiday import USFederalHolidayCalendar
fs = gcsfs.GCSFileSystem()

pd.set_option('display.max_columns', None)

In [3]:
GCS_FILE_PATH  = 'gs://calitp-analytics-data/data-analyses/ahsc_grant/ahsc_riderships'

In [4]:
bart_ridership = pd.read_excel(f"{GCS_FILE_PATH}/bart_ridership.xlsx")
big_blue_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/big_blue_bus_ridership.xlsx")
caltrain_ridership = pd.read_excel(f"{GCS_FILE_PATH}/caltrain_ridership.xlsx")
culver_citybus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/culver_citybus_ridership.xlsx")
foothill_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/foothill_transit_ridership.xlsx")
fresno_area_express_ridership = pd.read_excel(f"{GCS_FILE_PATH}/fresno_area_express_ridership.xlsx")
gold_coast_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/gold_coast_transit_ridership.xlsx")
golden_gate_park_shuttle_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_park_shuttle_ridership.xlsx")
golden_gate_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/golden_gate_transit_ridership.xlsx")
long_beach_transit_ridership=  pd.read_excel(f"{GCS_FILE_PATH}/long_beach_transit_ridership.xlsx")
octa_ridership = pd.read_excel(f"{GCS_FILE_PATH}/octa_ridership.xlsx")
omni_trans_ridership= pd.read_excel(f"{GCS_FILE_PATH}/omni_trans_ridership.xlsx")
riverside_transit_ridership= pd.read_excel(f"{GCS_FILE_PATH}/riverside_transit_ridership.xlsx")
sacrt_bus_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sacrt_bus_ridership.xlsx")
samtrans_ridership = pd.read_excel(f"{GCS_FILE_PATH}/samtrans_ridership.xlsx")
santa_cruz_metro_ridership = pd.read_excel(f"{GCS_FILE_PATH}/santa_cruz_metro_ridership.xlsx")
sbmtd_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sbmtd_ridership.xlsx")
sdmts_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sdmts_ridership.xlsx")
sunline_transit_ridership = pd.read_excel(f"{GCS_FILE_PATH}/sunline_transit_ridership.xlsx")


In [5]:
def add_day_type(df, date_col, output_col='day_type'):
    """
    Classify each row as 'Weekday', 'Saturday', or 'Sunday' based on a specified date column.
    This is useful when daily datasets include labels such as 'Holiday' or when no day_type
    column exists. The function converts the date column, identifies the day of week, and
    standardizes the day-type classification accordingly. Either a start_date or end_date
    column can be used for datasets with daily reporting.
    """
    df = df.copy()

    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

    dow = df[date_col].dt.dayofweek  

    df[output_col] = np.select(
        condlist=[dow.eq(5), dow.eq(6)],
        choicelist=['Saturday', 'Sunday'],
        default='Weekday'
    )

    return df

In [6]:
# Counting number of days without holidays
def count_service_days(row):
    
    if str(row.day_type).lower() == "all":
        return (row.end_date - row.start_date).days + 1

    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum(d.weekday() < 5 for d in dates)

    elif row.day_type == "Saturday":
        return sum(d.weekday() == 5 for d in dates)

    elif row.day_type == "Sunday":
        return sum(d.weekday() == 6 for d in dates)

In [7]:
# Counting number of days with holidays
def count_service_days_wo_holidays(row):
    dates = pd.date_range(row.start_date, row.end_date)

    if row.day_type == "Weekday":
        return sum((d.weekday() < 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Saturday":
        return sum((d.weekday() == 5) and (d not in holiday_set) for d in dates)

    elif row.day_type == "Sunday":
        return sum((d.weekday() == 6) and (d not in holiday_set) for d in dates)

In [8]:
def compute_averages(df, ridership_cols, group_cols):
    """
    Computes average of one or more ridership columns per group in long format.

    Parameters:
    - df: pandas DataFrame
    - ridership_cols: list of columns to average (e.g., ['daily_boardings', 'daily_alightings'])
    - group_cols: list of columns to group by (e.g., ['stop_name', 'day_type'])

    Returns:
    - DataFrame with columns: group_cols + average of each ridership column
    """
    
    if isinstance(ridership_cols, str):
        ridership_cols = [ridership_cols]  # allow single string input
    
    averaged = df.groupby(group_cols)[ridership_cols].mean().reset_index()
    
    # rename each column to indicate it's averaged
    averaged = averaged.rename(columns={col: f"average_{col}" for col in ridership_cols})
    
    return averaged

In [9]:
master_cols = [
    "organization_name",
    "stop_id",
    "stop_name",
    "stop_lat",
    "stop_lon",
    "day_type",
    "average_daily_boardings",
    "average_daily_alightings",
    "start_date",
    "end_date"
]

def standardize_columns(df, master_cols):
    
    # add missing columns
    for col in master_cols:
        if col not in df.columns:
            df[col] = None

    # keep only master columns and order them
    return df[master_cols]

In [10]:
bart_ridership = add_day_type(bart_ridership, date_col='start_date')

bart_ridership = bart_ridership.rename(columns={
    'Station Name': 'stop_name',
    'Stop ID': 'stop_id',
    'Date': 'date'
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_name",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_bart_ridership = (
    bart_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("Entries", "sum"),
        daily_alightings=("Exits", "sum")
    )
)

# Set the ridership basis flag
total_bart_ridership["daily_ridership_basis"] = "reported_daily"


In [11]:
# Calculate average ridership per day type and stop
total_bart_ridership['daily_boardings'] = pd.to_numeric(total_bart_ridership['daily_boardings'], errors='coerce')
total_bart_ridership['daily_alightings'] = pd.to_numeric(total_bart_ridership['daily_alightings'], errors='coerce')

average_bart_ridership = compute_averages(
    df=total_bart_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "day_type", "stop_name"]
)

average_bart_ridership["organization_name"] = "San Francisco Bay Area Rapid Transit District"

average_bart_ridership[['start_date','end_date']] = [bart_ridership['start_date'].min(), bart_ridership['end_date'].max()]
# Remove last digit from all stop_id
average_bart_ridership['stop_id'] = average_bart_ridership['stop_id'].astype(str).str[:-1]

bart_final = standardize_columns(average_bart_ridership, master_cols)

bart_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
123,San Francisco Bay Area Rapid Transit District,90630,San Bruno,None,None,Saturday,841.115385,861.826923,2024-10-01,2025-09-30
117,San Francisco Bay Area Rapid Transit District,90610,Colma,None,None,Saturday,842.557692,853.365385,2024-10-01,2025-09-30
7,San Francisco Bay Area Rapid Transit District,90030,MacArthur,None,None,Sunday,1737.961538,1738.153846,2024-10-01,2025-09-30
106,San Francisco Bay Area Rapid Transit District,90460,Richmond,None,None,Sunday,1144.307692,1091.173077,2024-10-01,2025-09-30
121,San Francisco Bay Area Rapid Transit District,90620,South San Francisco,None,None,Sunday,463.615385,486.596154,2024-10-01,2025-09-30


In [12]:
big_blue_bus_ridership.columns = big_blue_bus_ridership.columns.str.lower()
big_blue_bus_ridership['service_day'] = big_blue_bus_ridership['service_day'].str.strip().str.capitalize()


big_blue_bus_ridership = big_blue_bus_ridership.rename(columns={
    'service_day': 'day_type',
    'average_daily_boardings':'average_daily_boardings_period',
    'average_daily_alightings':'average_daily_alightings_period'    
})


# Sum boardings/alightings across directions within same stop & period
total_big_blue_bus_ridership = (
    big_blue_bus_ridership
    .groupby(['day_type',
              'stop_id','stop_name','stop_lat','stop_lon',
              'start_date','end_date'])
    .agg({
        'average_daily_boardings_period':'sum',
        'average_daily_alightings_period':'sum'
    })
    .reset_index()
)


In [13]:
big_blue_bus_ridership['start_date'] = pd.to_datetime(big_blue_bus_ridership['start_date'])
big_blue_bus_ridership['end_date'] = pd.to_datetime(big_blue_bus_ridership['end_date'])

total_big_blue_bus_ridership["n_days"] = total_big_blue_bus_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 4 different time periods
averaged_total_big_blue_bus_ridership = (
    total_big_blue_bus_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['average_daily_alightings_period'], weights=x['n_days'])
    }))
    .reset_index()
)


# Set the ridership basis flag
averaged_total_big_blue_bus_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_total_big_blue_bus_ridership["organization_name"] = "Big Blue Bus"

averaged_total_big_blue_bus_ridership[['start_date','end_date']] = [big_blue_bus_ridership['start_date'].min(), big_blue_bus_ridership['end_date'].max()]

big_blue_bus_ridership_final = standardize_columns(averaged_total_big_blue_bus_ridership, master_cols)
big_blue_bus_ridership_final.sample(5)

/tmp/ipykernel_4799/1528211661.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1074,Big Blue Bus,1283,OCEAN PARK EB/HIGHLAND NS,34.005446,-118.478765,Sunday,3.551048,2.390940,2024-08-01,2025-11-30
495,Big Blue Bus,2030,PICO WB/WESTWOOD NS,34.040330,-118.428377,Saturday,38.133964,77.870833,2024-08-01,2025-11-30
1485,Big Blue Bus,1000,4TH NB/COLORADO FS (Downtown SM Station),34.014235,-118.492457,Weekday,395.910224,488.908436,2024-08-01,2025-11-30
2155,Big Blue Bus,2267,SAN VICENTE WB/BRISTOL NS,34.049911,-118.485076,Weekday,0.007598,0.648157,2024-08-01,2025-11-30
2152,Big Blue Bus,2268,SAN VICENTE WB/AVONDALE NS,34.049004,-118.489617,Weekday,0.011026,5.074119,2024-08-01,2025-11-30


In [14]:
caltrain_ridership = caltrain_ridership.rename(columns={
    'Origin Station': 'stop_name',
    'Date Type': 'day_type',
    'Average Ridership':'average_daily_boardings_period',  
})

caltrain_ridership = caltrain_ridership[
    caltrain_ridership["day_type"] != "Holiday"
]

# Getting number of days and using US Federal Holidays
start_date_caltrain = '2023-11-01'
end_date_caltrain = '2025-07-31'

#Getting all the federal holidays between the start and end date
cal = USFederalHolidayCalendar()
holidays = cal.holidays(start=start_date_caltrain, end=end_date_caltrain)

# Setting holidays to calculate the number of weekdays, saturday and sunday without holidays 
holiday_set = set(holidays)

caltrain_ridership["start_date"] = pd.to_datetime(caltrain_ridership["start_date"])
caltrain_ridership["end_date"] = pd.to_datetime(caltrain_ridership["end_date"])

caltrain_ridership["n_days"] = caltrain_ridership.apply(count_service_days_wo_holidays, axis=1)

In [15]:
# Taking weighted average of the ridership across 20 months time periods
averaged_caltrain_ridership = (
    caltrain_ridership
    .groupby(['day_type','stop_name'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['average_daily_boardings_period'], weights=x['n_days'])
    }))
    .reset_index()
)

# Set the ridership basis flag
averaged_caltrain_ridership["daily_ridership_basis"] = "reported_avg_daily"
averaged_caltrain_ridership["organization_name"] = "Caltrain"

averaged_caltrain_ridership[['start_date','end_date']] = [caltrain_ridership['start_date'].min(), caltrain_ridership['end_date'].max()]

caltrain_ridership_final = standardize_columns(averaged_caltrain_ridership, master_cols)

caltrain_ridership_final.sample(5)

/tmp/ipykernel_4799/3722569284.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
75,Caltrain,None,Morgan Hill,None,None,Weekday,103.849297,None,2023-11-01,2025-07-31
4,Caltrain,None,Broadway,None,None,Saturday,80.031640,None,2023-11-01,2025-07-31
7,Caltrain,None,Capitol,None,None,Saturday,0.138475,None,2023-11-01,2025-07-31
47,Caltrain,None,Palo Alto,None,None,Sunday,916.209789,None,2023-11-01,2025-07-31
38,Caltrain,None,College Park,None,None,Sunday,2.643171,None,2023-11-01,2025-07-31


In [16]:
culver_citybus_ridership["daily_ridership_basis"] = "reported_avg_daily"



culver_citybus_ridership = culver_citybus_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

group_cols = [
     "stop_id", "stop_name", "day_type", "daily_ridership_basis"]

# Sum AVG On and AVG Off
sum_culver_citybus_ridership = (culver_citybus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("AVG On", "sum"),
        average_daily_alightings=("AVG Off", "sum")
    )
)

sum_culver_citybus_ridership["organization_name"] = "Culver City Bus"

sum_culver_citybus_ridership[['start_date','end_date']] = [culver_citybus_ridership['start_date'].min(), culver_citybus_ridership['end_date'].max()]

culver_citybus_final = standardize_columns(sum_culver_citybus_ridership, master_cols)

culver_citybus_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
158,Culver City Bus,154,WashingtonBlvd/OverlandAve,None,None,Weekday,72.8,49.5,2025-07-14,2025-08-25
360,Culver City Bus,342,Beverly Glen Blvd/Olympic Blvd,None,None,Saturday,1.9,5.5,2025-07-14,2025-08-25
261,Culver City Bus,302,Green Valley Cir/Sepulveda Blvd,None,None,Weekday,3.4,1.5,2025-07-14,2025-08-25
297,Culver City Bus,320,Overland Ave/Virginia Ave,None,None,Saturday,15.6,5.7,2025-07-14,2025-08-25
61,Culver City Bus,121,Washington Blvd/Corinth Ave,None,None,Sunday,3.8,7.1,2025-07-14,2025-08-25


In [17]:
# Add day_type from the date column
foothill_transit_ridership = add_day_type(foothill_transit_ridership, date_col="date")

foothill_transit_ridership = foothill_transit_ridership.rename(columns={
    'stop_code': 'stop_id',
})

# Grouping columns
group_cols = [
    "date", "stop_id", "stop_lat", "stop_lon",
    "start_date", "end_date", "day_type"
]

# Sum boardings and alightings, with explicit output names
total_ridership_foothill = (
    foothill_transit_ridership
    .groupby(group_cols, as_index=False)
    .agg(
        daily_boardings=("boardings", "sum"),
        daily_alightings=("alightings", "sum")
    )
)

# Set the ridership basis flag
total_ridership_foothill["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_ridership_foothill['daily_boardings'] = pd.to_numeric(total_ridership_foothill['daily_boardings'], errors='coerce')
total_ridership_foothill['daily_alightings'] = pd.to_numeric(total_ridership_foothill['daily_alightings'], errors='coerce')

average_foothill_ridership = compute_averages(
    df=total_ridership_foothill,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=[ "stop_id", 
                "stop_lat", "stop_lon", "day_type"]
)

average_foothill_ridership["organization_name"] = "Foothill Transit"

average_foothill_ridership[['start_date','end_date']] = [foothill_transit_ridership['start_date'].min(), foothill_transit_ridership['end_date'].max()]
foothill_final = standardize_columns(average_foothill_ridership, master_cols)

foothill_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
116,Foothill Transit,177,None,34.135968,-117.907122,Sunday,272.416667,38.138889,2024-07-01,2025-06-30
567,Foothill Transit,745,None,34.060796,-117.908077,Weekday,30.532567,8.356322,2024-07-01,2025-06-30
1308,Foothill Transit,1171,None,34.072235,-118.044371,Weekday,874.114943,435.547893,2024-07-01,2025-06-30
1005,Foothill Transit,975,None,34.092944,-117.706724,Sunday,2.900000,0.300000,2024-07-01,2025-06-30
285,Foothill Transit,594,None,34.049064,-117.968009,Weekday,12.019157,3.727969,2024-07-01,2025-06-30


In [18]:
# Add day_type from the date column
fresno_area_express_ridership = add_day_type(fresno_area_express_ridership, date_col="Date")

fresno_area_express_ridership["daily_ridership_basis"] = "reported_daily"


fresno_area_express_ridership = fresno_area_express_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopLabel': 'stop_name',
    'ProjectedBoarding': 'daily_boardings',
    'ProjectedAlighting': 'daily_alightings'
})

# Calculate average ridership per day type and stop
fresno_area_express_ridership['daily_boardings'] = pd.to_numeric(fresno_area_express_ridership['daily_boardings'], errors='coerce')
fresno_area_express_ridership['daily_alightings'] = pd.to_numeric(fresno_area_express_ridership['daily_alightings'], errors='coerce')

average_fresno_area_express_ridership = compute_averages(
    df=fresno_area_express_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id",  "stop_name", "day_type"]
)

average_fresno_area_express_ridership["organization_name"] = "Fresno County"

average_fresno_area_express_ridership[['start_date','end_date']] = [fresno_area_express_ridership['start_date'].min(), fresno_area_express_ridership['end_date'].max()]

fresno_final = standardize_columns(average_fresno_area_express_ridership, master_cols)

fresno_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3780,Fresno County,1759,NE VAN NESS - UNIVERSITY,None,None,Saturday,8.482180,9.382280,2024-09-01,2025-08-31
4019,Fresno County,1888,NW NEES - CALLISCH,None,None,Weekday,0.699737,2.282397,2024-09-01,2025-08-31
3166,Fresno County,1486,SE NEES - BOND,None,None,Sunday,1.092694,1.177562,2024-09-01,2025-08-31
664,Fresno County,343,SW SAN JOSE - MARTY,None,None,Saturday,22.142778,17.297479,2024-09-01,2025-08-31
265,Fresno County,144,FIRST - SHIELDS,None,None,Saturday,47.521306,70.416826,2024-09-01,2025-08-31


In [19]:
gold_coast_transit_ridership = gold_coast_transit_ridership.rename(columns={
    'day_of_week': 'day_type',
    'lat': 'stop_lat',
    'lon': 'stop_lon'
})

group_cols = ["day_type", "stop_id", "stop_name",  "stop_lat", "stop_lon", "start_date", "end_date"]

# Sum AVG On 
total_gold_coast_transit_ridership = (gold_coast_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("total_on", "sum"),
        daily_alightings=("total_off", "sum"),
    )
)

total_gold_coast_transit_ridership['start_date'] = pd.to_datetime(total_gold_coast_transit_ridership['start_date'])
total_gold_coast_transit_ridership['end_date'] = pd.to_datetime(total_gold_coast_transit_ridership['end_date'])

total_gold_coast_transit_ridership["n_days"] = total_gold_coast_transit_ridership.apply(count_service_days, axis=1)

In [20]:
# Taking weighted average of the ridership across 4 different time periods
averaged_gold_coast_transit_ridership = (
    total_gold_coast_transit_ridership
    .groupby(['day_type', 'stop_name',
              'stop_id','stop_lat','stop_lon'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['daily_boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['daily_alightings'], weights=x['n_days'])
    }))
    .reset_index()
)

averaged_gold_coast_transit_ridership["organization_name"] = "Gold Coast Transit"

averaged_gold_coast_transit_ridership[['start_date','end_date']] = [gold_coast_transit_ridership['start_date'].min(), gold_coast_transit_ridership['end_date'].max()]
golden_coast_transit_final = standardize_columns(averaged_gold_coast_transit_ridership, master_cols)

golden_coast_transit_final.sample(5)

/tmp/ipykernel_4799/1215055102.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
11,Gold Coast Transit,3RDGAR1,3rd & Garfield,34.200605,-119.174483,Weekday,3.0,7.0,2025-05-01,2025-05-31
101,Gold Coast Transit,CIBGIS1,Channel Isl & Gisler,34.173272,-119.172299,Weekday,12.0,5.0,2025-05-01,2025-05-31
473,Gold Coast Transit,TPHPET2,Telephone & Petit,34.272059,-119.171028,Weekday,52.0,20.0,2025-05-01,2025-05-31
540,Gold Coast Transit,VNALCL2,Ventura Ave & Los Cabos,34.329182,-119.291482,Weekday,4.0,1.0,2025-05-01,2025-05-31
492,Gold Coast Transit,THOCHE1,Thompson & Chestnut,34.278359,-119.291294,Weekday,5.0,15.0,2025-05-01,2025-05-31


In [21]:
# Add day_type from the date column
golden_gate_park_shuttle_ridership = add_day_type(golden_gate_park_shuttle_ridership, date_col="Date")

group_cols = [
    "Date", "day_type", "Stop", "Stop ID", "start_date", "end_date"]

# Sum AVG On 
total_golden_gate_park_shuttle_ridership = (golden_gate_park_shuttle_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("Ridership", "sum"),
    )
)

total_golden_gate_park_shuttle_ridership["daily_ridership_basis"] = "reported_daily"

total_golden_gate_park_shuttle_ridership = total_golden_gate_park_shuttle_ridership.rename(columns={
    'Stop': 'stop_name',
    'Date': 'date',
    'Stop ID': 'stop_id'
})

# Calculate average ridership per day type and stop
total_golden_gate_park_shuttle_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_park_shuttle_ridership['daily_boardings'], errors='coerce')
average_golden_gate_park_shuttle_ridership = compute_averages(
    df=total_golden_gate_park_shuttle_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_name", "day_type", "stop_id"]
)


average_golden_gate_park_shuttle_ridership["organization_name"] = "Golden Gate Park Shuttle"

average_golden_gate_park_shuttle_ridership[['start_date','end_date']] = [golden_gate_park_shuttle_ridership['start_date'].min(), 
                                                                         golden_gate_park_shuttle_ridership['end_date'].max()]
golden_gate_parkshuttle_final = standardize_columns(average_golden_gate_park_shuttle_ridership, master_cols)

golden_gate_parkshuttle_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
13,Golden Gate Park Shuttle,7609,Academy of Sciences,None,None,Sunday,14.903846,None,2024-07-01,2025-06-30
45,Golden Gate Park Shuttle,7615,Tennis Center/ Dalia Dell EB,None,None,Saturday,11.192308,None,2024-07-01,2025-06-30
6,Golden Gate Park Shuttle,7611,8th Ave EB,None,None,Saturday,10.673077,None,2024-07-01,2025-06-30
30,Golden Gate Park Shuttle,7619,JFK Gateway EB,None,None,Saturday,5.576923,None,2024-07-01,2025-06-30
40,Golden Gate Park Shuttle,7604,Rose Garden - EB,None,None,Sunday,15.192308,None,2024-07-01,2025-06-30


In [22]:
# Add day_type from the date column
golden_gate_transit_ridership = add_day_type(golden_gate_transit_ridership, date_col="date")

golden_gate_transit_ridership = golden_gate_transit_ridership.rename(columns={
    'STOP_NUMBER': 'stop_id',
    'STOP_NAME': 'stop_name',
})

# Remove everything in parentheses (including the parentheses) at the end of the string
golden_gate_transit_ridership['stop_name'] = golden_gate_transit_ridership['stop_name'].str.replace(r'\s*\(.*\)$', '', regex=True)

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
total_golden_gate_transit_ridership = (golden_gate_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        daily_boardings=("BOARDINGS", "sum"),
        daily_alightings=("ALIGHTINGS", "sum")
    )
)

total_golden_gate_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_golden_gate_transit_ridership['daily_boardings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_boardings'], errors='coerce')
total_golden_gate_transit_ridership['daily_alightings'] = pd.to_numeric(total_golden_gate_transit_ridership['daily_alightings'], errors='coerce')

average_golden_gate_transit_ridership = compute_averages(
    df=total_golden_gate_transit_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "day_type"]
)

average_golden_gate_transit_ridership["organization_name"] = "Golden Gate Transit"
average_golden_gate_transit_ridership[['start_date','end_date']] = [golden_gate_transit_ridership['start_date'].min(), 
                                                                         golden_gate_transit_ridership['end_date'].max()]

golden_gate_transit_final = standardize_columns(average_golden_gate_transit_ridership, master_cols)

golden_gate_transit_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
389,Golden Gate Transit,40949,Piner Rd & Industrial Dr,None,None,Saturday,31.250000,24.000000,2025-09-01,2025-09-30
588,Golden Gate Transit,42203,Mission St & 2nd St,None,None,Saturday,0.750000,21.000000,2025-09-01,2025-09-30
3,Golden Gate Transit,40006,Folsom St & 2nd St,None,None,Weekday,41.190476,8.714286,2025-09-01,2025-09-30
554,Golden Gate Transit,42161,Cutting Blvd & S 23rd St,None,None,Weekday,14.409091,4.863636,2025-09-01,2025-09-30
308,Golden Gate Transit,40699,DeLong Ave & Reichert Ave,None,None,Sunday,0.500000,7.250000,2025-09-01,2025-09-30


In [23]:
long_beach_transit_ridership["daily_ridership_basis"] = "reported_avg_daily"

group_cols = [
    "DayType", "StopID", "StopName", "end_date", "start_date", "daily_ridership_basis"]

# Sum AVG On and AVG Off
total_long_beach_transit_ridership = (long_beach_transit_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Boardings", "sum"),
        average_daily_alightings=("Alightings", "sum")
    )
)

total_long_beach_transit_ridership = total_long_beach_transit_ridership.rename(columns={
    'StopID': 'stop_id',
    'StopName': 'stop_name',
    'DayType': 'day_type',
})

total_long_beach_transit_ridership["organization_name"] = "Long Beach Transit"

total_long_beach_transit_ridership[['start_date','end_date']] = [long_beach_transit_ridership['start_date'].min(), 
                                                                         long_beach_transit_ridership['end_date'].max()]

long_beach_transit_final = standardize_columns(total_long_beach_transit_ridership, master_cols)

long_beach_transit_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3472,Long Beach Transit,4049,Via Oro & 3900 E,None,None,Sunday,0.000000,0.432008,2024-07-01,2025-06-30
5013,Long Beach Transit,1885,Wardlow & Claremore SE,None,None,Weekday,0.000000,8.801980,2024-07-01,2025-06-30
4850,Long Beach Transit,1695,PCH & Martin Luther King Jr NW,None,None,Weekday,128.297448,147.708031,2024-07-01,2025-06-30
1511,Long Beach Transit,2219,Gardendale & Barlin SE,None,None,Saturday,0.000000,3.264713,2024-07-01,2025-06-30
3881,Long Beach Transit,387,Magnolia & 19th NE,None,None,Weekday,12.114286,0.000000,2024-07-01,2025-06-30


In [24]:
octa_ridership = octa_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',

})

octa_ridership["stop_name"] = octa_ridership["stop_name"].str.split("-", n=1).str[1]

group_cols = [
    "day_type", "stop_id", "stop_name", "end_date", "start_date"]

# Sum AVG On and AVG Off
octa_ridership_sum = (octa_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("APC Boarding", "sum"),
        average_daily_alightings=("APC Alighting", "sum")
    )
)

octa_ridership_sum.loc[
    octa_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

octa_ridership_sum["organization_name"] = "Orange County Transportation Authority"

octa_ridership_sum[['start_date','end_date']] = [octa_ridership['start_date'].min(), 
                                                                         octa_ridership['end_date'].max()]

octa_final = standardize_columns(octa_ridership_sum, master_cols)

octa_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1189,Orange County Transportation Authority,1854,MAGNOLIA-OLIVE,None,None,Weekday,18,1,2025-02-04,2025-02-04
4094,Orange County Transportation Authority,6734,TUSTIN-17TH,None,None,Weekday,64,69,2025-02-04,2025-02-04
1288,Orange County Transportation Authority,2008,EDINGER-BUSHARD,None,None,Weekday,8,2,2025-02-04,2025-02-04
3623,Orange County Transportation Authority,6031,ALTON-HALLADAY,None,None,Weekday,5,4,2025-02-04,2025-02-04
829,Orange County Transportation Authority,1208,HARBOR-SCENIC,None,None,Weekday,5,11,2025-02-04,2025-02-04


In [25]:
omni_trans_ridership= omni_trans_ridership.rename(columns={
    'Stop Name': 'stop_name',
    'avg_boardings': 'average_daily_boardings',
    'avg_alightings': 'average_daily_alightings',
    'Stop Id': 'stop_id'
})

omni_trans_ridership["organization_name"] = "Omnitrans"

omni_trans_ridership_filtered = omni_trans_ridership[['stop_name', 'stop_id',  'average_daily_boardings', 'average_daily_alightings', 'organization_name', 'day_type']]

omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
    lambda x: None if pd.isna(x) else int(x)
)

omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(), 
                                                                         omni_trans_ridership['end_date'].max()]
omni_final = standardize_columns(omni_trans_ridership_filtered, master_cols)

omni_final.sample(5)

/tmp/ipykernel_4799/3069775595.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered['stop_id'] = omni_trans_ridership_filtered['stop_id'].apply(
/tmp/ipykernel_4799/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  omni_trans_ridership_filtered[['start_date','end_date']] = [omni_trans_ridership['start_date'].min(),
/tmp/ipykernel_4799/3069775595.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_in

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
3495,Omnitrans,5718.0,Highland @ Rockford,None,None,all,1.846575,0.816438,2023-07-01,2026-06-30
2732,Omnitrans,5652.0,Sierra @ Spring/Arrow,None,None,all,0.120548,1.013699,2023-07-01,2026-06-30
3367,Omnitrans,5476.0,E ST @ 14TH,None,None,all,0.147945,0.260274,2023-07-01,2026-06-30
2212,Omnitrans,6589.0,9th @ Central,None,None,all,1.175342,1.334247,2023-07-01,2026-06-30
2980,Omnitrans,5941.0,RIVERSIDE @ BENSON,None,None,all,6.868493,5.038356,2023-07-01,2026-06-30


In [26]:
riverside_transit_ridership = add_day_type(riverside_transit_ridership, date_col='date')

group_cols = [
    "date", "Stop ID", "start_date", 
    "end_date", "day_type"]

# Sum AVG On and AVG Off
total_riverside_transit_ridership = (riverside_transit_ridership.groupby(
    group_cols, as_index=False).agg(
    daily_boardings = ('ridership', 'sum'))
)

total_riverside_transit_ridership = total_riverside_transit_ridership.rename(columns={
    'Stop ID': 'stop_id'

})

In [27]:
total_riverside_transit_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
total_riverside_transit_ridership['daily_boardings'] = pd.to_numeric(total_riverside_transit_ridership['daily_boardings'], errors='coerce')
average_riverside_transit_ridership = compute_averages(
    df=total_riverside_transit_ridership,
    ridership_cols=["daily_boardings"],
    group_cols=["stop_id", "day_type"]
)

average_riverside_transit_ridership["organization_name"] = "Riverside Transit"

average_riverside_transit_ridership[['start_date','end_date']] = [riverside_transit_ridership['start_date'].min(), 
                                                                         riverside_transit_ridership['end_date'].max()]

riverside_final = standardize_columns(average_riverside_transit_ridership, master_cols)

riverside_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
763,Riverside Transit,1306,None,None,None,Weekday,3.255474,None,2025-01-01,2025-10-31
5032,Riverside Transit,3192,None,None,None,Weekday,1.372881,None,2025-01-01,2025-10-31
1777,Riverside Transit,1698,None,None,None,Saturday,1.750000,None,2025-01-01,2025-10-31
231,Riverside Transit,1095,None,None,None,Weekday,1.939024,None,2025-01-01,2025-10-31
3548,Riverside Transit,2411,None,None,None,Sunday,1.000000,None,2025-01-01,2025-10-31


In [28]:
group_cols = [
    "stop_id", "stop_lat", "stop_lon", "stop_name", "day_type"]

# Combining across directions
sacrt_bus_ridership_sum = (sacrt_bus_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("average_boardings", "sum"),
        average_daily_alightings=("average_alightings", "sum")
    )
)

sacrt_bus_ridership_sum.loc[
    sacrt_bus_ridership_sum['day_type'].str.strip().str.lower() == "weekday",
    'day_type'
] = "Weekday"

# Replace day_type
sacrt_bus_ridership_sum["day_type"] = sacrt_bus_ridership_sum["day_type"].replace("daily", "all")

sacrt_bus_ridership_sum["organization_name"] = "SacRT Bus"

sacrt_bus_ridership_sum[['start_date','end_date']] = [sacrt_bus_ridership['start_date'].min(), 
                                                                         sacrt_bus_ridership['end_date'].max()]

sacrt_bus_final = standardize_columns(sacrt_bus_ridership_sum, master_cols)

sacrt_bus_final.sample(5)


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1292,SacRT Bus,1697,J ST & 43RD ST (WB),38.568612,-121.449280,all,3,1,2023-09-01,2023-12-31
2808,SacRT Bus,5257,NORTH B ST & 12TH ST (WB),38.590398,-121.485208,Weekday,22,7,2023-09-01,2023-12-31
2362,SacRT Bus,3387,TWIN OAKS AVE & MARIPOSA AVE (WB),38.718533,-121.281067,all,1,2,2023-09-01,2023-12-31
768,SacRT Bus,1051,LAND PARK DR & CORDANO WAY (NB),38.550302,-121.498650,Weekday,0,1,2023-09-01,2023-12-31
2624,SacRT Bus,4124,53RD ST & 12TH AVE (SB),38.541165,-121.443982,Weekday,0,1,2023-09-01,2023-12-31


In [29]:
samtrans_ridership = add_day_type(samtrans_ridership, date_col='APC Date')

samtrans_ridership = samtrans_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Lat': 'stop_lat',
    'Lon': 'stop_lon',
    'APC Date': 'date',
    'Ons': 'daily_boardings',
    'Offs': 'daily_alightings',
})

samtrans_ridership["daily_ridership_basis"] = "reported_daily"

# Calculate average ridership per day type and stop
samtrans_ridership['daily_boardings'] = pd.to_numeric(samtrans_ridership['daily_boardings'], errors='coerce')
samtrans_ridership['daily_alightings'] = pd.to_numeric(samtrans_ridership['daily_alightings'], errors='coerce')

average_samtrans_ridership = compute_averages(
    df=samtrans_ridership,
    ridership_cols=["daily_boardings", "daily_alightings"],
    group_cols=["stop_id", "stop_name", "stop_lat", "stop_lon", "day_type"]
)

average_samtrans_ridership["organization_name"] = "Samtrans"

average_samtrans_ridership[['start_date','end_date']] = [samtrans_ridership['start_date'].min(), 
                                                                         samtrans_ridership['end_date'].max()]

samtrans_final = standardize_columns(average_samtrans_ridership, master_cols)

samtrans_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
1440,Samtrans,333511,Colma BART-Bay 11,37.685188,-122.464706,Weekday,74.904762,160.428571,2025-08-01,2025-08-31
2135,Samtrans,335648,College Dr & Susan Dr,37.632946,-122.454483,Saturday,0.400000,3.600000,2025-08-01,2025-08-31
1795,Samtrans,334533,Gellert Blvd & Sunrise Ct,37.641551,-122.448104,Sunday,0.600000,0.800000,2025-08-01,2025-08-31
3603,Samtrans,346039,Durham St & Laurel Ave,37.464573,-122.153728,Weekday,12.000000,0.200000,2025-08-01,2025-08-31
3809,Samtrans,347060,University Ave & Chaucer St,37.455753,-122.150978,Weekday,1.500000,0.250000,2025-08-01,2025-08-31


In [30]:
group_cols = [
    "Stop ID",  "Stop Name", "day_type", "start_date", 
    "end_date"]

# Sum AVG On and AVG Off
total_sdmts_ridership = (sdmts_ridership.groupby(
    group_cols, as_index=False).agg(
        average_daily_boardings=("Average On", "sum"),
        average_daily_alightings=("Average Off", "sum")
    )
)


total_sdmts_ridership = total_sdmts_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
})

total_sdmts_ridership["daily_ridership_basis"] = "reported_avg_daily"

total_sdmts_ridership["organization_name"] = "SDMTS"

total_sdmts_ridership[['start_date','end_date']] = [sdmts_ridership['start_date'].min(), 
                                                                         sdmts_ridership['end_date'].max()]

sdmts_final = standardize_columns(total_sdmts_ridership, master_cols)

sdmts_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
10036,SDMTS,99108,Encanto/62nd St Trolley Stn,None,None,Saturday,140.327402,73.048246,2024-09-01,2025-01-25
10282,SDMTS,99320,Paradise Valley Rd & Worthington St,None,None,Weekday,10.929701,26.881230,2024-09-01,2025-01-25
2708,SDMTS,11671,Park Bl & Howard Av,None,None,Sunday,16.734777,22.285915,2024-09-01,2025-01-25
7792,SDMTS,60068,National City Bl & 30th St,None,None,Saturday,2.581066,2.538167,2024-09-01,2025-01-25
7755,SDMTS,60042,Palm Av & Hollister St (Trolley),None,None,Weekday,279.772888,453.244955,2024-09-01,2025-01-25


In [31]:
santa_cruz_metro_ridership = santa_cruz_metro_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'Boardings': 'average_daily_boardings',
    'Alightings': 'average_daily_alightings'    
})

santa_cruz_metro_ridership["organization_name"] = "Santa Cruz Metro"

santa_cruz_metro_ridership[['start_date','end_date']] = [santa_cruz_metro_ridership['start_date'].min(), 
                                                                         santa_cruz_metro_ridership['end_date'].max()]

santa_cruz_final = standardize_columns(santa_cruz_metro_ridership, master_cols)

santa_cruz_final.sample(5)

,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
17,Santa Cruz Metro,2159,38th Ave + Tranquility Ct,None,None,all,5.846575,1.895890,2024-07-01,2025-06-30
625,Santa Cruz Metro,1741,Scotts Valley Dr + Victor Square,None,None,all,1.054795,1.616438,2024-07-01,2025-06-30
126,Santa Cruz Metro,2759,Bronte Ave + Ester Way (Bronte Park),None,None,all,2.572603,2.452055,2024-07-01,2025-06-30
769,Santa Cruz Metro,2661,Western Dr + Western Ct,None,None,all,1.216438,4.093151,2024-07-01,2025-06-30
713,Santa Cruz Metro,1846,Soquel Dr + W Ledyard Way,None,None,all,0.819178,0.517808,2024-07-01,2025-06-30


In [32]:
sbmtd_ridership = sbmtd_ridership.rename(columns={
    'Stop ID': 'stop_id',
    'Stop Name': 'stop_name',
    'avg_boardings': 'daily_boardings',
    'avg_ridership': 'daily_alightings'    
})


sbmtd_ridership["n_days"] = sbmtd_ridership.apply(count_service_days, axis=1)

# Taking weighted average of the ridership across 12 months
averaged_sbmtd_ridership = (
    sbmtd_ridership
    .groupby(['stop_id', 'stop_name', 'day_type'])
    .apply(lambda x: pd.Series({
        'average_daily_boardings': np.average(x['Ridership by Stop_Boardings'], weights=x['n_days']),
        'average_daily_alightings': np.average(x['Ridership by Stop_Alightings'], weights=x['n_days'])
    }))
    .reset_index()
)



averaged_sbmtd_ridership["organization_name"] = "SBMTD"

averaged_sbmtd_ridership[['start_date','end_date']] = [sbmtd_ridership['start_date'].min(), 
                                                                         sbmtd_ridership['end_date'].max()]

sbmtd_final = standardize_columns(averaged_sbmtd_ridership, master_cols)

sbmtd_final.sample(5)

/tmp/ipykernel_4799/1734702637.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: pd.Series({


,organization_name,stop_id,stop_name,stop_lat,stop_lon,day_type,average_daily_boardings,average_daily_alightings,start_date,end_date
400,SBMTD,477,Cathedral Oaks & Ribera,None,None,all,0.084932,3.441096,2024-11-01,2025-10-31
202,SBMTD,247,Storke & Santa Felicia,None,None,all,87.334247,308.273973,2024-11-01,2025-10-31
277,SBMTD,325,Lillie & Colville,None,None,all,56.813699,34.808219,2024-11-01,2025-10-31
353,SBMTD,428,Abrego & Camino Pescadero,None,None,all,728.663014,689.109589,2024-11-01,2025-10-31
161,SBMTD,201,Milpas & Puerto Vallarta,None,None,all,73.073973,201.687671,2024-11-01,2025-10-31


In [33]:
all_ridership = pd.concat([
    bart_final,
    big_blue_bus_ridership_final,
    caltrain_ridership_final,
    culver_citybus_final,
    foothill_final,
    fresno_final,
    golden_coast_transit_final,
    golden_gate_parkshuttle_final,
    golden_gate_transit_final,
    long_beach_transit_final,
    octa_final,
    omni_final,
    riverside_final,
    sacrt_bus_final,
    samtrans_final,
    sdmts_final,
    santa_cruz_final,
    sbmtd_final,
], ignore_index=True)

/tmp/ipykernel_4799/1180593662.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_ridership = pd.concat([


In [34]:
all_ridership.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55521 entries, 0 to 55520
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   organization_name         55521 non-null  object        
 1   stop_id                   55111 non-null  object        
 2   stop_name                 44301 non-null  object        
 3   stop_lat                  15342 non-null  float64       
 4   stop_lon                  15342 non-null  float64       
 5   day_type                  55521 non-null  object        
 6   average_daily_boardings   55521 non-null  float64       
 7   average_daily_alightings  49139 non-null  float64       
 8   start_date                55521 non-null  datetime64[ns]
 9   end_date                  55521 non-null  datetime64[ns]
dtypes: datetime64[ns](2), float64(4), object(4)
memory usage: 4.2+ MB


In [35]:
all_ridership.to_csv(f"{GCS_FILE_PATH}/AHSC_2026/ridership_all.csv", index=False)